In [3]:
from google.colab import auth
auth.authenticate_user()

In [9]:
import os
os.environ["GOOGLE_CLOUD_PROJECT"] = "tts-eval-489108"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "tts-eval-489108-14cd0038a031.json"
os.environ["GOOGLE_CLOUD_PROJECT"] = "tts-eval-489108"

In [10]:
TESTS = [
    ("hi-IN", "hi", "native", "general", "आज मौसम साफ है और हम खेत में काम कर रहे हैं।"),
    ("hi-IN", "hi", "native", "agri", "गेहूं की फसल में पीले धब्बे दिख रहे हैं, क्या दवा डालनी चाहिए?"),
    ("hi-IN", "hi", "native", "numbers", "आज की तारीख 27/02/2026 है और मेरा मोबाइल नंबर 9876543210 है।"),
    ("hi-IN", "hi", "roman",  "roman_numbers", "Aaj ki tareekh 27/02/2026 hai aur mera mobile number 9876543210 hai."),

    ("pa-IN", "pa", "native", "general", "ਅੱਜ ਮੌਸਮ ਚੰਗਾ ਹੈ ਅਤੇ ਅਸੀਂ ਖੇਤ ਵਿੱਚ ਕੰਮ ਕਰ ਰਹੇ ਹਾਂ।"),
    ("pa-IN", "pa", "native", "numbers", "ਅੱਜ ਦੀ ਤਾਰੀਖ 27/02/2026 ਹੈ ਤੇ ਮੇਰਾ ਮੋਬਾਈਲ ਨੰਬਰ 9876543210 ਹੈ।"),
    ("pa-IN", "pa", "roman",  "roman_numbers", "Ajj di tareekh 27/02/2026 ae te mera mobile number 9876543210 ae."),

    ("bn-IN", "bn", "native", "general", "আজ আবহাওয়া ভালো এবং আমরা মাঠে কাজ করছি।"),
    ("bn-IN", "bn", "native", "numbers", "আজকের তারিখ 27/02/2026 এবং আমার মোবাইল নম্বর 9876543210।"),
    ("bn-IN", "bn", "roman",  "roman_numbers", "Aajker tarikh 27/02/2026 ebong amar mobile number 9876543210."),

    ("gu-IN", "gu", "native", "general", "આજે હવામાન સારું છે અને અમે ખેતરમાં કામ કરી રહ્યા છીએ।"),
    ("gu-IN", "gu", "native", "numbers", "આજની તારીખ 27/02/2026 છે અને મારો મોબાઇલ નંબર 9876543210 છે।"),
    ("gu-IN", "gu", "roman",  "roman_numbers", "Aajni tarik 27/02/2026 chhe ane maro mobile number 9876543210 chhe."),

    ("ta-IN", "ta", "native", "general", "இன்று வானிலை நல்லதாக உள்ளது, நாங்கள் வயலில் வேலை செய்கிறோம்."),
    ("ta-IN", "ta", "native", "numbers", "இன்றைய தேதி 27/02/2026 மற்றும் என் கைபேசி எண் 9876543210."),
    ("ta-IN", "ta", "roman",  "roman_numbers", "Inraiya thethi 27/02/2026; en mobile number 9876543210."),

    ("hi-IN", "mix", "mixed", "hinglish", "Kal mandi rate check karna hai, please update kar dena."),
]

In [11]:
from google.cloud import texttospeech

client = texttospeech.TextToSpeechClient()
voices = client.list_voices()

print("Connected successfully.")
print("Total voices available:", len(voices.voices))

Connected successfully.
Total voices available: 2066


In [12]:
from google.cloud import texttospeech

client = texttospeech.TextToSpeechClient()

TARGET_LANGS = sorted(set([t[0] for t in TESTS]))
voices = client.list_voices().voices

def is_standard(v):
    # Standard voices typically have names like: "xx-XX-Standard-A"
    return "Standard" in v.name

standard_by_lang = {lang: [] for lang in TARGET_LANGS}

for v in voices:
    if not is_standard(v):
        continue
    for lang in v.language_codes:
        if lang in standard_by_lang:
            standard_by_lang[lang].append(v)

for lang, vlist in standard_by_lang.items():
    print(f"\n{lang} -> {len(vlist)} Standard voices")
    for v in vlist[:10]:
        print("  ", v.name, str(v.ssml_gender).split(".")[-1])
    if len(vlist) > 10:
        print("   ...")


bn-IN -> 4 Standard voices
   bn-IN-Standard-A 2
   bn-IN-Standard-B 1
   bn-IN-Standard-C 2
   bn-IN-Standard-D 1

gu-IN -> 4 Standard voices
   gu-IN-Standard-A 2
   gu-IN-Standard-B 1
   gu-IN-Standard-C 2
   gu-IN-Standard-D 1

hi-IN -> 6 Standard voices
   hi-IN-Standard-A 2
   hi-IN-Standard-B 1
   hi-IN-Standard-C 1
   hi-IN-Standard-D 2
   hi-IN-Standard-E 2
   hi-IN-Standard-F 1

pa-IN -> 4 Standard voices
   pa-IN-Standard-A 2
   pa-IN-Standard-B 1
   pa-IN-Standard-C 2
   pa-IN-Standard-D 1

ta-IN -> 4 Standard voices
   ta-IN-Standard-A 2
   ta-IN-Standard-B 1
   ta-IN-Standard-C 2
   ta-IN-Standard-D 1


In [13]:
from google.cloud.texttospeech import SsmlVoiceGender

picked_voice = {}
for lang, vlist in standard_by_lang.items():
    pref = [SsmlVoiceGender.NEUTRAL, SsmlVoiceGender.FEMALE, SsmlVoiceGender.MALE]
    chosen = None
    for g in pref:
        for v in vlist:
            if v.ssml_gender == g:
                chosen = v
                break
        if chosen:
            break
    if not chosen and vlist:
        chosen = vlist[0]
    picked_voice[lang] = chosen.name if chosen else None

print("Picked Standard voices:")
for lang in TARGET_LANGS:
    print(lang, "->", picked_voice[lang])

Picked Standard voices:
bn-IN -> bn-IN-Standard-A
gu-IN -> gu-IN-Standard-A
hi-IN -> hi-IN-Standard-A
pa-IN -> pa-IN-Standard-A
ta-IN -> ta-IN-Standard-A


In [14]:
import time, os, pandas as pd
from pathlib import Path

OUT_DIR = Path("/content/tts_outputs/standard")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def synthesize_standard(lang_code: str, text: str, voice_name: str, speaking_rate=1.0, pitch=0.0):
    voice = texttospeech.VoiceSelectionParams(
        language_code=lang_code,
        name=voice_name,
    )
    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3,
        speaking_rate=speaking_rate,
        pitch=pitch,
    )
    inp = texttospeech.SynthesisInput(text=text)

    t0 = time.perf_counter()
    resp = client.synthesize_speech(input=inp, voice=voice, audio_config=audio_config)
    t1 = time.perf_counter()

    return resp.audio_content, (t1 - t0)

rows = []
for (lang_code, lang_tag, script_type, case_type, text) in TESTS:
    voice_name = picked_voice.get(lang_code)
    if not voice_name:
        rows.append({
            "model_family": "Standard",
            "lang_code": lang_code,
            "case": case_type,
            "script": script_type,
            "voice": None,
            "latency_sec": None,
            "file": None,
            "note": "No Standard voice available in this project"
        })
        continue

    audio, latency = synthesize_standard(lang_code, text, voice_name)

    fname = f"standard__{lang_code}__{script_type}__{case_type}.mp3".replace("/", "-")
    fpath = OUT_DIR / fname
    with open(fpath, "wb") as f:
        f.write(audio)

    rows.append({
        "model_family": "Standard",
        "lang_code": lang_code,
        "case": case_type,
        "script": script_type,
        "voice": voice_name,
        "latency_sec": round(latency, 3),
        "file": str(fpath),
        "note": ""
    })

df_standard = pd.DataFrame(rows)
df_standard

,model_family,lang_code,case,script,voice,latency_sec,file,note
0,Standard,hi-IN,general,native,hi-IN-Standard-A,0.204,/content/tts_outputs/standard/standard__hi-IN_...,
1,Standard,hi-IN,agri,native,hi-IN-Standard-A,0.297,/content/tts_outputs/standard/standard__hi-IN_...,
2,Standard,hi-IN,numbers,native,hi-IN-Standard-A,0.331,/content/tts_outputs/standard/standard__hi-IN_...,
3,Standard,hi-IN,roman_numbers,roman,hi-IN-Standard-A,0.374,/content/tts_outputs/standard/standard__hi-IN_...,
4,Standard,pa-IN,general,native,pa-IN-Standard-A,0.199,/content/tts_outputs/standard/standard__pa-IN_...,
5,Standard,pa-IN,numbers,native,pa-IN-Standard-A,0.234,/content/tts_outputs/standard/standard__pa-IN_...,
6,Standard,pa-IN,roman_numbers,roman,pa-IN-Standard-A,0.177,/content/tts_outputs/standard/standard__pa-IN_...,
7,Standard,bn-IN,general,native,bn-IN-Standard-A,0.237,/content/tts_outputs/standard/standard__bn-IN_...,
8,Standard,bn-IN,numbers,native,bn-IN-Standard-A,0.350,/content/tts_outputs/standard/standard__bn-IN_...,
9,Standard,bn-IN,roman_numbers,roman,bn-IN-Standard-A,0.344,/content/tts_outputs/standard/standard__bn-IN_...,
